# Vibecoded vs Not-Vibecoded — Training Notebook

This notebook does the following, in order:

1. Mount Google Drive and pull your zipped, preprocessed dataset onto the Colab VM's local disk (fast I/O for training).
2. Load `processeddata.csv` and split it into **train / validation / test**, grouped by `source_site` (so the same website never appears in two different splits — this prevents the model from "cheating" by memorizing a site it already saw in training).
3. Build a pretrained ResNet, replace its final layer for 2-class output.
4. **Phase 1**: freeze the backbone, train only the new classifier head.
5. **Phase 2**: unfreeze the last block, fine-tune with a much smaller learning rate.
6. Evaluate on the held-out test set (accuracy, confusion matrix, per-class metrics).
7. Save + download the final model weights.

### About surviving Colab disconnects
Every epoch, training progress (model weights, optimizer state, current epoch, best validation accuracy so far) is saved to **Google Drive**, not the Colab VM's local disk (local disk is wiped when your runtime resets). If you get disconnected:

1. Reconnect, then **Runtime -> Run all** (or just re-run the cells top to bottom).
2. Drive gets remounted, the zip gets re-extracted locally (fast, a few seconds/minutes), and then the training cells will **automatically detect your last saved checkpoint and resume from the next epoch** instead of starting over.

You don't need to do anything special beyond re-running the notebook.

### Before you start
Zip your **`processed_images/`** folder (the one with `0/` and `1/` subfolders from the preprocessing script) together with `processeddata.csv`, so the zip looks like:

```
processed_images.zip
|-- processed_images/
|   |-- 0/   (not vibecoded)
|   `-- 1/   (vibecoded)
`-- processeddata.csv
```

Upload that zip **once** to your Google Drive, e.g. to `MyDrive/vibecoded_project/processed_images.zip`. You will NOT need to re-upload it again even if Colab disconnects -- it stays on Drive.

> Note: ignore the `path` column inside the CSV -- it was written on your local machine and won't be valid in Colab. We reconstruct the correct path ourselves from `label` + `filename`.


## Step 1 -- Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


## Step 2 -- Configuration

Edit the paths in `drive_zip_path` and `drive_project_dir` to match where you uploaded your zip on Drive.
Everything else (checkpoints, logs, split file, final model) will be saved under `drive_project_dir` automatically.


In [ ]:
import os

CONFIG = {
    # --- paths (EDIT THESE) ---
    "drive_zip_path": "/content/drive/MyDrive/vibecoded_project/processed_images.zip",
    "drive_project_dir": "/content/drive/MyDrive/vibecoded_project",   # checkpoints/logs/model saved here
    "local_data_dir": "/content/local_data",                           # fast local disk on the Colab VM

    # --- dataset ---
    "csv_name": "processeddata.csv",
    "image_size": 224,
    "num_classes": 2,
    "class_names": ["not_vibecoded", "vibecoded"],   # label 0, label 1

    # --- split ---
    "test_frac": 0.15,
    "val_frac": 0.15,     # fraction of the FULL dataset, not of the remainder
    "split_seed": 42,

    # --- dataloaders ---
    "batch_size": 32,
    "num_workers": 2,

    # --- phase 1: train the head only (backbone frozen) ---
    "phase1_lr": 1e-3,
    "phase1_epochs": 5,

    # --- phase 2: fine-tune last block (backbone mostly frozen, layer4 unfrozen) ---
    "phase2_lr": 1e-5,
    "phase2_epochs": 10,

    # --- logging ---
    "log_every_n_batches": 20,
}

# derived paths
CONFIG["checkpoint_dir"] = os.path.join(CONFIG["drive_project_dir"], "checkpoints")
CONFIG["logs_dir"] = os.path.join(CONFIG["drive_project_dir"], "logs")
CONFIG["split_file"] = os.path.join(CONFIG["drive_project_dir"], "split_assignment.csv")
CONFIG["final_model_dir"] = os.path.join(CONFIG["drive_project_dir"], "final_model")

for d in [CONFIG["checkpoint_dir"], CONFIG["logs_dir"], CONFIG["final_model_dir"], CONFIG["local_data_dir"]]:
    os.makedirs(d, exist_ok=True)

print("Config ready. Checkpoints/logs will persist at:", CONFIG["drive_project_dir"])


## Step 3 -- Unzip dataset onto local disk

We copy the zip from Drive to the Colab VM's local disk and extract it there. Training reads
thousands of small image files per epoch -- doing that directly from Drive is noticeably slower
than from local disk, so this local copy is worth doing every session.

This step is idempotent: if you re-run this cell in the same session and the data's already there, it skips re-extracting.


In [ ]:
import zipfile, time

local_extract_marker = os.path.join(CONFIG["local_data_dir"], ".extracted_ok")

if os.path.exists(local_extract_marker):
    print("Data already extracted locally this session, skipping.")
else:
    assert os.path.exists(CONFIG["drive_zip_path"]), f"Zip not found at {CONFIG['drive_zip_path']} -- check the path in CONFIG."
    t0 = time.time()
    print(f"Extracting {CONFIG['drive_zip_path']} -> {CONFIG['local_data_dir']} ...")
    with zipfile.ZipFile(CONFIG["drive_zip_path"], "r") as zf:
        zf.extractall(CONFIG["local_data_dir"])
    with open(local_extract_marker, "w") as f:
        f.write("ok")
    print(f"Done in {time.time() - t0:.1f}s")

# sanity check the expected structure exists
img_root = os.path.join(CONFIG["local_data_dir"], "processed_images")
csv_path = os.path.join(CONFIG["local_data_dir"], CONFIG["csv_name"])
assert os.path.isdir(img_root), f"Expected folder not found: {img_root}"
assert os.path.exists(csv_path), f"Expected CSV not found: {csv_path}"
print("Found:", img_root)
print("Found:", csv_path)


## Step 4 -- Imports, seeds, device

In [ ]:
import random
import json as jsonlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
from PIL import Image

SEED = CONFIG["split_seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU.")


## Step 5 -- Load the CSV and inspect it

Quick sanity-check logs so you can see class balance and device balance before training on it.


In [ ]:
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from {csv_path}")
print()
print("Label counts:")
print(df["label"].value_counts())
print()
if "device" in df.columns:
    print("Device counts:")
    print(df["device"].value_counts())
    print()
if "source_site" in df.columns:
    n_sites = df["source_site"].nunique()
    print(f"Unique source sites: {n_sites}")
df.head()


## Step 6 -- Grouped, stratified train / val / test split

We split by **`source_site`**, not by individual image. If we split by image instead, the same
website (e.g. `7minti_mobile_002` and `7minti_desktop_002`) could end up in both train and test,
letting the model partially memorize that specific site rather than learning general patterns --
this would make test accuracy look better than it really is.

The split is computed once and saved to `split_assignment.csv` on Drive. On every future run,
we reuse that saved split instead of recomputing it -- this guarantees train/val/test membership
never changes between sessions, even across disconnects.


In [ ]:
if os.path.exists(CONFIG["split_file"]):
    print("Existing split file found, reusing it (guarantees consistency across sessions).")
    split_df = pd.read_csv(CONFIG["split_file"])
else:
    print("No split file found yet, computing a new one...")

    # one row per site, with that site's label (every image from a site shares the same label,
    # since sites live entirely inside one class folder)
    site_labels = df.groupby("source_site")["label"].agg(lambda s: s.mode().iloc[0]).reset_index()
    site_labels.columns = ["source_site", "label"]

    sites_trainval, sites_test = train_test_split(
        site_labels["source_site"],
        test_size=CONFIG["test_frac"],
        stratify=site_labels["label"],
        random_state=SEED,
    )

    trainval_labels = site_labels.set_index("source_site").loc[sites_trainval, "label"]
    val_frac_of_trainval = CONFIG["val_frac"] / (1 - CONFIG["test_frac"])
    sites_train, sites_val = train_test_split(
        sites_trainval,
        test_size=val_frac_of_trainval,
        stratify=trainval_labels,
        random_state=SEED,
    )

    split_map = {}
    for s in sites_train: split_map[s] = "train"
    for s in sites_val:   split_map[s] = "val"
    for s in sites_test:  split_map[s] = "test"

    split_df = pd.DataFrame(list(split_map.items()), columns=["source_site", "split"])
    split_df.to_csv(CONFIG["split_file"], index=False)
    print(f"Split file saved to {CONFIG['split_file']}")

df = df.merge(split_df, on="source_site", how="left")
assert df["split"].isna().sum() == 0, "Some rows have no split assignment -- check for sites missing from split_assignment.csv"

print()
print("Images per split:")
print(df["split"].value_counts())
print()
print("Label balance per split (should be roughly balanced in each):")
print(df.groupby("split")["label"].value_counts())


## Step 7 -- Dataset and DataLoaders

Images were already resized + letterboxed to a fixed square during preprocessing, so the only
transform needed here is converting to a tensor and normalizing with ImageNet's mean/std --
required because we're using an ImageNet-pretrained ResNet, whose learned features expect
input in that exact normalization range.

As requested, **no data augmentation** is applied for now.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),  # safety net, images are already this size
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class VibecodedDataset(Dataset):
    def __init__(self, dataframe, image_root, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # reconstruct path locally: processed_images/<label>/<filename>
        img_path = os.path.join(self.image_root, str(int(row["label"])), row["filename"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        label = int(row["label"])
        return image, label

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

train_ds = VibecodedDataset(train_df, img_root, transform)
val_ds   = VibecodedDataset(val_df, img_root, transform)
test_ds  = VibecodedDataset(test_df, img_root, transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"train: {len(train_ds)} images, {len(train_loader)} batches")
print(f"val:   {len(val_ds)} images, {len(val_loader)} batches")
print(f"test:  {len(test_ds)} images, {len(test_loader)} batches")

# quick visual sanity check: show one batch
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i in range(6):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)  # undo normalization for display
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(CONFIG["class_names"][labels[i].item()])
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## Step 8 -- Model: pretrained ResNet18

We use **ResNet18** pretrained on ImageNet. It's a good default for a dataset this size (~10k
images): enough capacity to learn meaningful visual patterns, small enough to fine-tune quickly
and avoid overfitting. If you later collect a lot more data, ResNet50 is a reasonable upgrade.

The final fully-connected layer (`fc`), originally built for ImageNet's 1000 classes, is replaced
with a fresh `Linear` layer outputting 2 classes.


In [ ]:
def build_model(num_classes=CONFIG["num_classes"]):
    model = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    # freeze the entire pretrained backbone to start
    for param in model.parameters():
        param.requires_grad = False

    # replace the classifier head -- this new layer is trainable by default
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

def unfreeze_last_block(model):
    # unfreeze layer4 (the last residual block) for phase-2 fine-tuning
    for param in model.layer4.parameters():
        param.requires_grad = True
    return model

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

model = build_model().to(DEVICE)
trainable, total = count_trainable_params(model)
print(f"Model built. Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## Step 9 -- Checkpoint save/load helpers

A "checkpoint" here means: model weights, optimizer state, which epoch we finished, and the
best validation accuracy seen so far -- everything needed to resume training exactly where it
left off. We save this to **Drive** after every single epoch, so at most one epoch of progress
is ever at risk from a disconnect.


In [ ]:
def save_checkpoint(path, model, optimizer, epoch, best_val_acc):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_acc": best_val_acc,
    }, path)

def load_checkpoint(path, model, optimizer, device):
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt

def load_history(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return jsonlib.load(f)
    return []

def save_history(path, history):
    with open(path, "w") as f:
        jsonlib.dump(history, f, indent=2)


## Step 10 -- The training loop function

This one function is reused for both Phase 1 and Phase 2. Key behavior:

- At the start, it checks for an existing checkpoint for this phase. If found, it resumes from
  `epoch + 1` instead of epoch 0 -- **this is what makes disconnects non-fatal.**
- Logs every `log_every_n_batches` batches (loss, running accuracy, elapsed time) so you can see
  it's actually progressing, not stalled.
- Logs a full summary at the end of every epoch (train loss/acc, val loss/acc, time taken).
- Saves a checkpoint after every epoch (`latest_*.pt`) and separately saves the best-performing
  model so far by validation accuracy (`best_*.pt`).
- Appends to a JSON history file after every epoch, so training curves survive disconnects too.


In [ ]:
def run_training_phase(model, train_loader, val_loader, optimizer, criterion,
                        num_epochs, phase_name, device):
    latest_ckpt_path = os.path.join(CONFIG["checkpoint_dir"], f"latest_{phase_name}.pt")
    best_ckpt_path    = os.path.join(CONFIG["checkpoint_dir"], f"best_{phase_name}.pt")
    history_path      = os.path.join(CONFIG["logs_dir"], f"history_{phase_name}.json")

    ckpt = load_checkpoint(latest_ckpt_path, model, optimizer, device)
    if ckpt is not None:
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        print(f"[{phase_name}] Resuming from checkpoint: starting at epoch {start_epoch+1}/{num_epochs}, "
              f"best_val_acc so far = {best_val_acc:.4f}")
    else:
        start_epoch = 0
        best_val_acc = 0.0
        print(f"[{phase_name}] No checkpoint found, starting fresh.")

    history = load_history(history_path)

    if start_epoch >= num_epochs:
        print(f"[{phase_name}] Checkpoint shows this phase is already complete ({start_epoch}/{num_epochs} epochs). Skipping.")
        return model, history, best_val_acc

    for epoch in range(start_epoch, num_epochs):
        epoch_start = time.time()

        # ---- train ----
        model.train()
        running_loss, running_correct, seen = 0.0, 0, 0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            seen += images.size(0)

            if (batch_idx + 1) % CONFIG["log_every_n_batches"] == 0:
                elapsed = time.time() - epoch_start
                print(f"  [{phase_name}][epoch {epoch+1}/{num_epochs}] "
                      f"batch {batch_idx+1}/{len(train_loader)} | "
                      f"loss={running_loss/seen:.4f} acc={running_correct/seen:.4f} | "
                      f"elapsed={elapsed:.1f}s")

        train_loss = running_loss / seen
        train_acc = running_correct / seen

        # ---- validate ----
        model.eval()
        val_loss_sum, val_correct, val_seen = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_seen += images.size(0)

        val_loss = val_loss_sum / val_seen
        val_acc = val_correct / val_seen
        epoch_time = time.time() - epoch_start

        print(f"[{phase_name}] EPOCH {epoch+1}/{num_epochs} SUMMARY | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
              f"time={epoch_time:.1f}s")

        history.append({
            "phase": phase_name, "epoch": epoch + 1,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "time_seconds": epoch_time,
        })
        save_history(history_path, history)

        # always save "latest" so we can resume
        save_checkpoint(latest_ckpt_path, model, optimizer, epoch, max(best_val_acc, val_acc))

        # separately track the best model by validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(best_ckpt_path, model, optimizer, epoch, best_val_acc)
            print(f"  -> new best {phase_name} model saved (val_acc={best_val_acc:.4f})")

    print(f"[{phase_name}] Finished all {num_epochs} epochs. Best val_acc = {best_val_acc:.4f}")
    return model, history, best_val_acc


## Step 11 -- Phase 1: train the classifier head (backbone frozen)

Only the new `fc` layer has trainable weights right now -- everything else is frozen, so there's
zero risk of damaging the pretrained ImageNet features at this stage.

**Learning rate: relatively high (`1e-3`).** This layer's weights are freshly, randomly
initialized (unlike the rest of the network, which already contains useful learned features), so
it needs bigger updates to move from "random" to "useful" in a reasonable number of epochs. There's
no danger in using a higher LR here specifically because the pretrained backbone is frozen and can't
be disturbed by it.


In [ ]:
model = build_model().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer_phase1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["phase1_lr"],
)

model, history_phase1, best_val_acc_phase1 = run_training_phase(
    model, train_loader, val_loader, optimizer_phase1, criterion,
    num_epochs=CONFIG["phase1_epochs"], phase_name="phase1", device=DEVICE,
)


## Step 12 -- Phase 2: unfreeze `layer4` and fine-tune

Now we unfreeze the last residual block (`layer4`) and continue training it together with the
classifier head.

**Learning rate: much lower (`1e-5`).** `layer4` already contains useful, pretrained features
(edges -> textures -> shapes -> higher-level patterns is roughly how CNNs build up representations,
and the later layers hold the more task-relevant ones). A large learning rate here would apply
big, disruptive updates and risk **catastrophic forgetting** -- wrecking those useful pretrained
features before the model has a chance to gently adapt them to your specific task. Small, careful
updates let it specialize without losing what it already knows.

We reload the best Phase 1 weights first, so Phase 2 fine-tunes from the best checkpoint of Phase 1
rather than whatever state Phase 1 happened to end on.


In [ ]:
# start Phase 2 from Phase 1's BEST checkpoint (not necessarily its last epoch)
best_phase1_path = os.path.join(CONFIG["checkpoint_dir"], "best_phase1.pt")
ckpt = torch.load(best_phase1_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded best phase1 model (val_acc={ckpt['best_val_acc']:.4f}) as the starting point for phase 2.")

model = unfreeze_last_block(model).to(DEVICE)
trainable, total = count_trainable_params(model)
print(f"Trainable params now: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

optimizer_phase2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["phase2_lr"],
)

model, history_phase2, best_val_acc_phase2 = run_training_phase(
    model, train_loader, val_loader, optimizer_phase2, criterion,
    num_epochs=CONFIG["phase2_epochs"], phase_name="phase2", device=DEVICE,
)


## A note on hyperparameter search

You asked whether a random search over hyperparameters is needed. For fine-tuning a pretrained
model on a dataset of this size (~10k images), it usually isn't necessary -- good defaults (Adam,
the two learning rates above, batch size 32) get you most of the way there, and an extensive
search is more likely to overfit your small validation set than meaningfully help.

If, after looking at the training curves below, validation accuracy plateaus early or looks
unstable, the first two things worth trying by hand (not a full search) are: a smaller
`phase2_lr` (e.g. `5e-6`), or more `phase1_epochs` before unfreezing anything. Only reach for a
real search (e.g. trying `phase2_lr` in `[1e-4, 1e-5, 1e-6]`) if manual adjustment isn't enough.


## Step 13 -- Plot training curves

In [ ]:
history_path1 = os.path.join(CONFIG["logs_dir"], "history_phase1.json")
history_path2 = os.path.join(CONFIG["logs_dir"], "history_phase2.json")
h1 = load_history(history_path1)
h2 = load_history(history_path2)
full_history = h1 + h2

epochs_all = list(range(1, len(full_history) + 1))
train_acc = [h["train_acc"] for h in full_history]
val_acc   = [h["val_acc"] for h in full_history]
train_loss = [h["train_loss"] for h in full_history]
val_loss   = [h["val_loss"] for h in full_history]
phase1_len = len(h1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_all, train_acc, label="train acc")
axes[0].plot(epochs_all, val_acc, label="val acc")
axes[0].axvline(phase1_len + 0.5, color="gray", linestyle="--", label="phase1 -> phase2")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("epoch (combined across both phases)")
axes[0].legend()

axes[1].plot(epochs_all, train_loss, label="train loss")
axes[1].plot(epochs_all, val_loss, label="val loss")
axes[1].axvline(phase1_len + 0.5, color="gray", linestyle="--", label="phase1 -> phase2")
axes[1].set_title("Loss")
axes[1].set_xlabel("epoch (combined across both phases)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Best val_acc phase1: {max(h['val_acc'] for h in h1):.4f}" if h1 else "no phase1 history")
print(f"Best val_acc phase2: {max(h['val_acc'] for h in h2):.4f}" if h2 else "no phase2 history")


## Step 14 -- Final evaluation on the test set

We pick whichever of `best_phase1` / `best_phase2` scored higher on validation, load it, and
evaluate ONLY NOW on the test set (which the model has never seen in any form during training or
model selection).


In [ ]:
best_phase1_path = os.path.join(CONFIG["checkpoint_dir"], "best_phase1.pt")
best_phase2_path = os.path.join(CONFIG["checkpoint_dir"], "best_phase2.pt")

ckpt1 = torch.load(best_phase1_path, map_location=DEVICE)
ckpt2 = torch.load(best_phase2_path, map_location=DEVICE)

if ckpt2["best_val_acc"] >= ckpt1["best_val_acc"]:
    chosen_ckpt, chosen_name = ckpt2, "phase2 (fine-tuned)"
else:
    chosen_ckpt, chosen_name = ckpt1, "phase1 (head only)"

print(f"Using best checkpoint from: {chosen_name}  (val_acc={chosen_ckpt['best_val_acc']:.4f})")

final_model = build_model().to(DEVICE)
if chosen_name.startswith("phase2"):
    final_model = unfreeze_last_block(final_model)
final_model.load_state_dict(chosen_ckpt["model_state"])
final_model.eval()

def get_probs_and_labels(model, loader, device):
    """Returns P(class=1, i.e. 'vibecoded') for every image in the loader, plus true labels."""
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]   # P(vibecoded)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_probs), np.array(all_labels)

# get probabilities on val (for threshold tuning, next step) and test (for final reporting)
val_probs, val_labels_arr = get_probs_and_labels(final_model, val_loader, DEVICE)
test_probs, test_labels_arr = get_probs_and_labels(final_model, test_loader, DEVICE)

# baseline: the standard default threshold of 0.5 (equivalent to plain argmax)
all_labels = test_labels_arr
all_preds = (test_probs >= 0.5).astype(int)

test_acc = accuracy_score(all_labels, all_preds)
print(f"\nTEST ACCURACY (default threshold = 0.5): {test_acc:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=CONFIG["class_names"]))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(CONFIG["class_names"])
ax.set_yticks([0, 1]); ax.set_yticklabels(CONFIG["class_names"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_title("Confusion matrix (test set, threshold=0.5)")
plt.tight_layout()
plt.show()


## Step 15 -- Tune the classification threshold (using the VALIDATION set)

By default the model predicts "vibecoded" whenever `P(vibecoded) >= 0.5`. That cutoff isn't
necessarily the one that gives the best F1 score -- depending on how confident/uncertain the
model tends to be for each class, a different threshold can do better.

**Important: this sweep uses the validation set, not the test set.** If we picked whichever
threshold maximizes F1 directly on the test set, we'd effectively be fitting a (small) model
parameter to the test data -- that inflates the reported score and stops the test set from being
a fair, untouched measure of real-world performance. Tuning on validation and then applying the
chosen threshold once to test keeps the evaluation honest.

We sweep a reasonable range of cutoffs and pick the one that maximizes F1 for the "vibecoded"
class.


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.30, 0.71, 0.05)
sweep_results = []
for t in thresholds:
    preds_at_t = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels_arr, preds_at_t, pos_label=1, zero_division=0)
    precision = precision_score(val_labels_arr, preds_at_t, pos_label=1, zero_division=0)
    recall = recall_score(val_labels_arr, preds_at_t, pos_label=1, zero_division=0)
    sweep_results.append({"threshold": round(float(t), 2), "f1": f1, "precision": precision, "recall": recall})

sweep_df = pd.DataFrame(sweep_results)
print("Threshold sweep on the VALIDATION set:")
print(sweep_df.to_string(index=False))

best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
best_threshold = float(best_row["threshold"])
print(f"\nBest threshold by F1 (on validation): {best_threshold}  "
      f"(F1={best_row['f1']:.4f}, precision={best_row['precision']:.4f}, recall={best_row['recall']:.4f})")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_df["threshold"], sweep_df["f1"], marker="o", label="F1")
ax.plot(sweep_df["threshold"], sweep_df["precision"], marker="o", label="precision")
ax.plot(sweep_df["threshold"], sweep_df["recall"], marker="o", label="recall")
ax.axvline(best_threshold, color="gray", linestyle="--", label=f"chosen threshold = {best_threshold}")
ax.set_xlabel("threshold")
ax.set_ylabel("score")
ax.set_title("Precision / Recall / F1 vs threshold (validation set)")
ax.legend()
plt.tight_layout()
plt.show()


## Step 16 -- Final test evaluation at the tuned threshold

We now apply `best_threshold` (chosen using only the validation set, never the test set) to the
test set, and compare it against the default-threshold (0.5) result from Step 14.


In [ ]:
all_preds_tuned = (test_probs >= best_threshold).astype(int)

test_acc_tuned = accuracy_score(all_labels, all_preds_tuned)
print(f"TEST ACCURACY  threshold={best_threshold}:  {test_acc_tuned:.4f}")
print(f"TEST ACCURACY  threshold=0.5 (baseline):  {test_acc:.4f}\n")

print(f"Classification report at threshold={best_threshold}:")
print(classification_report(all_labels, all_preds_tuned, target_names=CONFIG["class_names"]))

cm_tuned = confusion_matrix(all_labels, all_preds_tuned)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm_tuned, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(CONFIG["class_names"])
ax.set_yticks([0, 1]); ax.set_yticklabels(CONFIG["class_names"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm_tuned[i, j], ha="center", va="center",
                 color="white" if cm_tuned[i, j] > cm_tuned.max()/2 else "black")
ax.set_title(f"Confusion matrix (test set, threshold={best_threshold})")
plt.tight_layout()
plt.show()

# from here on, treat the tuned predictions as THE predictions
all_preds = all_preds_tuned


## Step 17 -- Look at a few misclassified test images

Useful for a sanity check: do the mistakes look like genuinely ambiguous/borderline cases, or
does the model seem confused for a systematic reason (e.g. always wrong on mobile screenshots)?
These are the misclassifications under the TUNED threshold.


In [ ]:
mistakes = [i for i, (p, l) in enumerate(zip(all_preds, all_labels)) if p != l]
print(f"{len(mistakes)} / {len(all_labels)} test images misclassified.")

n_show = min(6, len(mistakes))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3.5))
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, mistakes[:n_show]):
        img_tensor, true_label = test_ds[idx]
        img = img_tensor.permute(1, 2, 0).numpy()
        img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"true: {CONFIG['class_names'][true_label]}\npred: {CONFIG['class_names'][all_preds[idx]]}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes to show (or none found in this pass).")


## Step 18 -- Save and download the final model

The final weights get saved to Drive (as a permanent backup) and also packaged into a small zip
you can download directly to your own computer, along with a metadata file describing how to
correctly load and use the model outside Colab (image size, normalization stats, class order,
architecture, and -- importantly -- the tuned classification threshold, since it's meaningless
without also knowing which cutoff to apply to the model's output probability).


In [ ]:
import shutil

final_state_path = os.path.join(CONFIG["final_model_dir"], "final_model_state_dict.pt")
torch.save(final_model.state_dict(), final_state_path)

metadata = {
    "architecture": "resnet18",
    "chosen_from": chosen_name,
    "num_classes": CONFIG["num_classes"],
    "class_names": CONFIG["class_names"],
    "image_size": CONFIG["image_size"],
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "classification_threshold": best_threshold,
    "test_accuracy_default_threshold_0.5": test_acc,
    "test_accuracy_tuned_threshold": test_acc_tuned,
}
metadata_path = os.path.join(CONFIG["final_model_dir"], "metadata.json")
with open(metadata_path, "w") as f:
    jsonlib.dump(metadata, f, indent=2)

print("Saved to Drive:")
print(" -", final_state_path)
print(" -", metadata_path)

# zip it up for a direct browser download
download_zip_path = "/content/final_model_package.zip"
shutil.make_archive("/content/final_model_package", "zip", CONFIG["final_model_dir"])

from google.colab import files
files.download(download_zip_path)
print("Download triggered -- check your browser downloads.")


## Loading the model later, outside Colab

To reuse `final_model_state_dict.pt` in a separate script:

```python
import torch, torchvision, torch.nn as nn, json

with open("metadata.json") as f:
    meta = json.load(f)

model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, meta["num_classes"])
model.load_state_dict(torch.load("final_model_state_dict.pt", map_location="cpu"))
model.eval()

# preprocess new images with the SAME image_size / mean / std stored in metadata.json, then:
with torch.no_grad():
    output = model(image_tensor)                      # image_tensor: shape [1, 3, H, W]
    prob_vibecoded = torch.softmax(output, dim=1)[0, 1].item()

# use the TUNED threshold saved in metadata.json -- NOT a plain argmax / 0.5 cutoff
prediction = "vibecoded" if prob_vibecoded >= meta["classification_threshold"] else "not_vibecoded"
```

## Suggestions for next steps

- **Data augmentation** -- you said to skip it for now, which is reasonable as a first baseline. If train accuracy climbs much higher than validation accuracy (a growing gap in the plots above), that's a sign of overfitting, and light augmentation (small brightness/contrast jitter -- avoid flips, since flipped UI text is meaningless) is usually the first fix to reach for.
- **ResNet50** -- worth trying once this baseline works, mainly if you gather more data; with only ~10k images ResNet18 is less prone to overfitting.
- **Grad-CAM** -- a visualization technique that highlights which regions of an image most influenced the model's decision. Genuinely useful here for sanity-checking that the model is picking up on layout/design cues rather than something spurious (e.g. a watermark or a specific font that happens to correlate with your data source).
- **Per-device breakdown** -- split the test accuracy by `device` (mobile vs desktop) to check the model isn't much weaker on one than the other.
- **Class-balance monitoring as you grow the dataset** -- keep `label` counts roughly equal as you add more sites, or use `class_weight`/weighted loss if that drifts.
